<a href="https://colab.research.google.com/github/dimitarpg13/agentic_architectures_and_design_patterns/blob/main/notebooks/multi_agent_comm_via_mcp/coordinator_specialized_workers_mcp_design_patterns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coordinator - Specialized Workers via MCP Design Pattern (Conceptual Simulation)

> ⚠️ **Important Disclaimer:** This notebook is a **conceptual simulation** that illustrates the *architectural pattern* of a coordinator with specialized worker agents communicating through a shared message bus. While the pattern is inspired by the ideas behind the **Model Context Protocol (MCP)**, the implementation here does **not** use the actual MCP protocol. Instead, it uses plain Python in-memory data structures (lists and dicts) to simulate the communication patterns that MCP would provide. The `mcp` package is imported but its server, client, and transport layers are not used.

## What is MCP?

The Model Context Protocol is an open protocol developed by Anthropic that standardizes how applications provide context to LLMs. It enables:
- **Standardized tool interfaces** between agents
- **Resource sharing** (data, context, state)
- **Prompts and templates** that can be shared
- **Interoperability** between different AI systems

## What This Notebook Actually Demonstrates

This notebook simulates the **communication patterns** that MCP enables, using simplified Python constructs:
1. **A custom `MessageBus`** (plain Python list/dict) simulates MCP-style message passing and resource sharing
2. **Agent classes** simulate MCP clients/servers but communicate via direct Python method calls
3. **A coordinator** orchestrates the workflow by calling agents sequentially — no real MCP transport is involved
4. **Tool registration** is demonstrated via a plain Python dictionary, not via MCP's tool protocol

To build a real MCP-based multi-agent system, you would use `mcp.server.Server` to expose tools/resources, connect via `mcp.client` over stdio or SSE transport, and use JSON-RPC for communication.

## Installation

```bash
pip install mcp anthropic httpx asyncio
```

In [1]:
# Import required libraries
import asyncio
import json
from typing import Any, Dict, List, Optional
from dataclasses import dataclass, field
from datetime import datetime
import uuid

In [2]:
!pip install mcp anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.9/397.9 kB 10.0 MB/s eta 0:00:00


## Part 1: Simulated Message Bus (Conceptual MCP Stand-in)

First, we create a `MessageBus` class that **simulates** the kind of message passing and resource sharing that MCP would provide. Note: this is a plain Python in-memory implementation — it does not use MCP's actual server, transport, or JSON-RPC protocol. The imported `mcp` types (`Server`, `Tool`, `Resource`, etc.) are not used in the communication logic.

In [ ]:
# NOTE: The mcp imports below are present for reference but are NOT actually used
# in the communication logic of this notebook. All agent communication goes through
# the custom MessageBus class (plain Python lists and dicts), not through MCP protocol.
from mcp.server import Server
from mcp.types import Tool, Resource, TextContent, ImageContent, EmbeddedResource
from mcp.server.stdio import stdio_server

# Shared message bus for inter-agent communication
# This is a SIMULATED message bus — NOT an MCP server or transport.
# In a real MCP system, this would be replaced by actual MCP server/client connections.
class MessageBus:
    """Simulates MCP-style message passing using plain Python in-memory data structures.
    This is NOT an actual MCP implementation."""
    def __init__(self):
        self.messages: List[Dict[str, Any]] = []
        self.resources: Dict[str, Any] = {}

    def publish(self, agent_id: str, message_type: str, content: Any):
        """Publish a message to the bus"""
        msg = {
            "id": str(uuid.uuid4()),
            "agent_id": agent_id,
            "type": message_type,
            "content": content,
            "timestamp": datetime.now().isoformat()
        }
        self.messages.append(msg)
        print(f"📤 [{agent_id}] Published: {message_type}")
        return msg["id"]

    def subscribe(self, agent_id: str, message_type: Optional[str] = None) -> List[Dict]:
        """Subscribe to messages from the bus"""
        if message_type:
            filtered = [m for m in self.messages
                       if m["type"] == message_type and m["agent_id"] != agent_id]
        else:
            filtered = [m for m in self.messages if m["agent_id"] != agent_id]
        return filtered

    def store_resource(self, resource_id: str, data: Any):
        """Store a shared resource"""
        self.resources[resource_id] = data
        print(f"💾 Stored resource: {resource_id}")

    def get_resource(self, resource_id: str) -> Optional[Any]:
        """Retrieve a shared resource"""
        return self.resources.get(resource_id)

# Global message bus
message_bus = MessageBus()

## Part 2: Agent Base Class (Simulated MCP-Style Interface)

Each agent wraps the `MessageBus` to publish/subscribe to messages and share resources. While the class is named `MCPAgent` and has an `expose_tool` method that creates an MCP `Tool` object, **no actual MCP server or client is instantiated** — all communication is via direct Python method calls on the shared `MessageBus`. The `expose_tool` method is dead code (never called in this notebook).

In [ ]:
@dataclass
class MCPAgent:
    """Base agent class that communicates via the simulated MessageBus.
    Despite the 'MCP' name, this class does NOT use actual MCP protocol —
    it wraps the in-memory MessageBus for publish/subscribe and resource sharing."""
    agent_id: str
    role: str
    capabilities: List[str] = field(default_factory=list)
    message_bus: MessageBus = field(default_factory=lambda: message_bus)

    def __post_init__(self):
        # Register agent on the message bus
        self.message_bus.publish(
            self.agent_id,
            "agent_registered",
            {"role": self.role, "capabilities": self.capabilities}
        )

    def expose_tool(self, tool_name: str, tool_func) -> Tool:
        """Creates an MCP Tool object, but this method is NEVER CALLED in this notebook.
        It's included to show what tool exposure could look like conceptually."""
        return Tool(
            name=f"{self.agent_id}_{tool_name}",
            description=f"Tool from {self.agent_id}: {tool_name}",
            inputSchema={
                "type": "object",
                "properties": {},
            }
        )

    def publish_result(self, result_type: str, data: Any):
        """Publish results to other agents via MCP"""
        return self.message_bus.publish(self.agent_id, result_type, data)

    def get_messages(self, message_type: Optional[str] = None) -> List[Dict]:
        """Get messages from other agents"""
        return self.message_bus.subscribe(self.agent_id, message_type)

    def share_resource(self, resource_name: str, data: Any):
        """Share a resource with other agents"""
        resource_id = f"{self.agent_id}_{resource_name}"
        self.message_bus.store_resource(resource_id, data)
        self.publish_result("resource_shared", {"resource_id": resource_id})
        return resource_id

    def access_resource(self, resource_id: str) -> Optional[Any]:
        """Access a shared resource"""
        return self.message_bus.get_resource(resource_id)

    async def process_task(self, task: str) -> Dict[str, Any]:
        """Process a task - to be overridden by specific agents"""
        raise NotImplementedError("Subclasses must implement process_task")

## Part 3: Specialized Agents with Simulated MCP-Style Communication

The specialized agents below (data collector, analyzer, report generator, validator) communicate through the shared `MessageBus` — **not** through actual MCP protocol calls. Each agent publishes results and reads messages from others using plain Python list/dict operations on the bus.

In [5]:
class DataCollectorAgent(MCPAgent):
    """Agent that collects and shares data"""

    def __init__(self):
        super().__init__(
            agent_id="data_collector",
            role="Data Collection",
            capabilities=["web_scraping", "api_calls", "data_extraction"]
        )

    async def process_task(self, task: str) -> Dict[str, Any]:
        """Simulate data collection"""
        print(f"\n🔍 [{self.agent_id}] Collecting data for: {task}")

        # Simulate data collection
        await asyncio.sleep(0.5)

        data = {
            "task": task,
            "data_points": [10, 25, 30, 45, 50],
            "source": "simulated_api",
            "timestamp": datetime.now().isoformat()
        }

        # Share data via MCP resource
        resource_id = self.share_resource("collected_data", data)

        # Publish completion message
        self.publish_result("data_collected", {
            "resource_id": resource_id,
            "summary": f"Collected {len(data['data_points'])} data points"
        })

        return data


class AnalysisAgent(MCPAgent):
    """Agent that analyzes data from other agents"""

    def __init__(self):
        super().__init__(
            agent_id="analyzer",
            role="Data Analysis",
            capabilities=["statistical_analysis", "pattern_recognition", "insights"]
        )

    async def process_task(self, task: str) -> Dict[str, Any]:
        """Analyze data from other agents"""
        print(f"\n📊 [{self.agent_id}] Analyzing data for: {task}")

        # Check for data collection messages
        messages = self.get_messages("data_collected")

        if not messages:
            print(f"⚠️  No data available for analysis")
            return {"error": "No data available"}

        # Get the latest data resource
        latest_msg = messages[-1]
        resource_id = latest_msg["content"]["resource_id"]
        data = self.access_resource(resource_id)

        print(f"📥 Retrieved resource: {resource_id}")

        # Simulate analysis
        await asyncio.sleep(0.5)

        analysis = {
            "task": task,
            "data_source": resource_id,
            "mean": sum(data["data_points"]) / len(data["data_points"]),
            "max": max(data["data_points"]),
            "min": min(data["data_points"]),
            "insights": "Data shows an upward trend",
            "timestamp": datetime.now().isoformat()
        }

        # Share analysis results
        resource_id = self.share_resource("analysis_results", analysis)

        # Publish completion
        self.publish_result("analysis_complete", {
            "resource_id": resource_id,
            "summary": f"Mean: {analysis['mean']:.2f}, Trend: Upward"
        })

        return analysis


class ReportGeneratorAgent(MCPAgent):
    """Agent that generates reports from analysis"""

    def __init__(self):
        super().__init__(
            agent_id="report_generator",
            role="Report Generation",
            capabilities=["report_writing", "visualization", "documentation"]
        )

    async def process_task(self, task: str) -> Dict[str, Any]:
        """Generate report from analysis"""
        print(f"\n📝 [{self.agent_id}] Generating report for: {task}")

        # Check for analysis completion messages
        messages = self.get_messages("analysis_complete")

        if not messages:
            print(f"⚠️  No analysis available for report")
            return {"error": "No analysis available"}

        # Get the analysis resource
        latest_msg = messages[-1]
        resource_id = latest_msg["content"]["resource_id"]
        analysis = self.access_resource(resource_id)

        print(f"📥 Retrieved resource: {resource_id}")

        # Simulate report generation
        await asyncio.sleep(0.5)

        report = {
            "title": f"Analysis Report: {task}",
            "summary": f"Analysis shows data with mean of {analysis['mean']:.2f}",
            "details": {
                "statistics": {
                    "mean": analysis["mean"],
                    "max": analysis["max"],
                    "min": analysis["min"]
                },
                "insights": analysis["insights"]
            },
            "timestamp": datetime.now().isoformat()
        }

        # Share final report
        resource_id = self.share_resource("final_report", report)

        # Publish completion
        self.publish_result("report_generated", {
            "resource_id": resource_id,
            "title": report["title"]
        })

        return report


class ValidationAgent(MCPAgent):
    """Agent that validates outputs from other agents"""

    def __init__(self):
        super().__init__(
            agent_id="validator",
            role="Validation",
            capabilities=["quality_check", "verification", "compliance"]
        )

    async def process_task(self, task: str) -> Dict[str, Any]:
        """Validate report quality"""
        print(f"\n✅ [{self.agent_id}] Validating outputs for: {task}")

        # Get all messages to validate the workflow
        all_messages = self.get_messages()

        validation = {
            "task": task,
            "workflow_complete": False,
            "steps_validated": [],
            "issues": []
        }

        # Check for required steps
        required_steps = ["data_collected", "analysis_complete", "report_generated"]

        for step in required_steps:
            step_messages = [m for m in all_messages if m["type"] == step]
            if step_messages:
                validation["steps_validated"].append(step)
                print(f"  ✓ {step} validated")
            else:
                validation["issues"].append(f"Missing step: {step}")
                print(f"  ✗ {step} missing")

        validation["workflow_complete"] = len(validation["steps_validated"]) == len(required_steps)

        # Publish validation results
        self.publish_result("validation_complete", validation)

        return validation

## Part 4: Workflow Coordinator (Simulated MCP Orchestration)

The coordinator orchestrates the multi-agent workflow by calling each agent's `process_task()` method sequentially. Despite the `MCP` prefix in the class name, it does **not** use MCP protocol — it simply iterates over a list of agent IDs and invokes them via direct Python calls.

In [ ]:
class MCPCoordinator:
    """Coordinates multi-agent workflows by calling agents sequentially.
    Despite the 'MCP' name, orchestration is done via direct Python method calls,
    not via MCP protocol."""

    def __init__(self):
        self.agents: Dict[str, MCPAgent] = {}
        self.message_bus = message_bus

    def register_agent(self, agent: MCPAgent):
        """Register an agent with the coordinator"""
        self.agents[agent.agent_id] = agent
        print(f"✅ Registered agent: {agent.agent_id} ({agent.role})")

    async def execute_workflow(self, task: str, agent_sequence: List[str]):
        """Execute a multi-agent workflow"""
        print(f"\n{'='*80}")
        print(f"🚀 Starting MCP Multi-Agent Workflow")
        print(f"📋 Task: {task}")
        print(f"🔄 Agent Sequence: {' → '.join(agent_sequence)}")
        print(f"{'='*80}\n")

        results = {}

        for agent_id in agent_sequence:
            if agent_id not in self.agents:
                print(f"❌ Agent {agent_id} not found")
                continue

            agent = self.agents[agent_id]
            result = await agent.process_task(task)
            results[agent_id] = result

            # Small delay to simulate processing time
            await asyncio.sleep(0.3)

        print(f"\n{'='*80}")
        print(f"✨ Workflow Complete")
        print(f"{'='*80}\n")

        return results

    def get_message_history(self) -> List[Dict]:
        """Get all messages from the bus"""
        return self.message_bus.messages

    def get_shared_resources(self) -> Dict[str, Any]:
        """Get all shared resources"""
        return self.message_bus.resources

    def display_communication_log(self):
        """Display the MCP communication log"""
        print("\n📡 MCP Communication Log:")
        print("="*80)

        for msg in self.message_bus.messages:
            timestamp = msg["timestamp"].split("T")[1].split(".")[0]
            print(f"[{timestamp}] {msg['agent_id']:20} | {msg['type']:25} | {str(msg['content'])[:50]}")

        print("="*80)

## Part 5: Running the Multi-Agent System

Let's create agents and execute workflows using MCP communication.

In [7]:
# Create the coordinator
coordinator = MCPCoordinator()

# Create and register agents
data_collector = DataCollectorAgent()
analyzer = AnalysisAgent()
report_gen = ReportGeneratorAgent()
validator = ValidationAgent()

coordinator.register_agent(data_collector)
coordinator.register_agent(analyzer)
coordinator.register_agent(report_gen)
coordinator.register_agent(validator)

📤 [data_collector] Published: agent_registered
📤 [analyzer] Published: agent_registered
📤 [report_generator] Published: agent_registered
📤 [validator] Published: agent_registered
✅ Registered agent: data_collector (Data Collection)
✅ Registered agent: analyzer (Data Analysis)
✅ Registered agent: report_generator (Report Generation)
✅ Registered agent: validator (Validation)


### Example 1: Complete Data Pipeline

In [8]:
# Execute a complete workflow
task = "Quarterly Sales Analysis"
agent_sequence = ["data_collector", "analyzer", "report_generator", "validator"]

results = await coordinator.execute_workflow(task, agent_sequence)


🚀 Starting MCP Multi-Agent Workflow
📋 Task: Quarterly Sales Analysis
🔄 Agent Sequence: data_collector → analyzer → report_generator → validator


🔍 [data_collector] Collecting data for: Quarterly Sales Analysis
💾 Stored resource: data_collector_collected_data
📤 [data_collector] Published: resource_shared
📤 [data_collector] Published: data_collected

📊 [analyzer] Analyzing data for: Quarterly Sales Analysis
📥 Retrieved resource: data_collector_collected_data
💾 Stored resource: analyzer_analysis_results
📤 [analyzer] Published: resource_shared
📤 [analyzer] Published: analysis_complete

📝 [report_generator] Generating report for: Quarterly Sales Analysis
📥 Retrieved resource: analyzer_analysis_results
💾 Stored resource: report_generator_final_report
📤 [report_generator] Published: resource_shared
📤 [report_generator] Published: report_generated

✅ [validator] Validating outputs for: Quarterly Sales Analysis
  ✓ data_collected validated
  ✓ analysis_complete validated
  ✓ report_generated 

In [9]:
# Display the communication log
coordinator.display_communication_log()


📡 MCP Communication Log:
[10:13:54] data_collector       | agent_registered          | {'role': 'Data Collection', 'capabilities': ['web_
[10:13:54] analyzer             | agent_registered          | {'role': 'Data Analysis', 'capabilities': ['statis
[10:13:54] report_generator     | agent_registered          | {'role': 'Report Generation', 'capabilities': ['re
[10:13:54] validator            | agent_registered          | {'role': 'Validation', 'capabilities': ['quality_c
[10:14:01] data_collector       | resource_shared           | {'resource_id': 'data_collector_collected_data'}
[10:14:01] data_collector       | data_collected            | {'resource_id': 'data_collector_collected_data', '
[10:14:02] analyzer             | resource_shared           | {'resource_id': 'analyzer_analysis_results'}
[10:14:02] analyzer             | analysis_complete         | {'resource_id': 'analyzer_analysis_results', 'summ
[10:14:03] report_generator     | resource_shared           | {'resource_id': 

In [10]:
# View shared resources
print("\n💾 Shared Resources via MCP:")
print("="*80)
for resource_id, data in coordinator.get_shared_resources().items():
    print(f"\n📦 Resource: {resource_id}")
    print(json.dumps(data, indent=2))


💾 Shared Resources via MCP:

📦 Resource: data_collector_collected_data
{
  "task": "Quarterly Sales Analysis",
  "data_points": [
    10,
    25,
    30,
    45,
    50
  ],
  "source": "simulated_api",
  "timestamp": "2026-02-03T10:14:01.430373"
}

📦 Resource: analyzer_analysis_results
{
  "task": "Quarterly Sales Analysis",
  "data_source": "data_collector_collected_data",
  "mean": 32.0,
  "max": 50,
  "min": 10,
  "insights": "Data shows an upward trend",
  "timestamp": "2026-02-03T10:14:02.232041"
}

📦 Resource: report_generator_final_report
{
  "title": "Analysis Report: Quarterly Sales Analysis",
  "summary": "Analysis shows data with mean of 32.00",
  "details": {
    "statistics": {
      "mean": 32.0,
      "max": 50,
      "min": 10
    },
    "insights": "Data shows an upward trend"
  },
  "timestamp": "2026-02-03T10:14:03.033624"
}


### Example 2: Parallel Agent Execution

This example demonstrates agents working in parallel using `asyncio.gather`. The parallelism here is standard Python async — it does not rely on MCP. In a real MCP system, parallel agents could run as separate MCP server processes.

In [11]:
async def parallel_data_collection():
    """Multiple data collectors working in parallel"""

    # Create multiple data collectors
    collector1 = DataCollectorAgent()
    collector1.agent_id = "data_collector_1"

    collector2 = DataCollectorAgent()
    collector2.agent_id = "data_collector_2"

    # Run in parallel
    tasks = [
        collector1.process_task("Dataset A"),
        collector2.process_task("Dataset B")
    ]

    results = await asyncio.gather(*tasks)

    print("\n✅ Parallel collection complete!")
    return results

# Run parallel collection
parallel_results = await parallel_data_collection()

📤 [data_collector] Published: agent_registered
📤 [data_collector] Published: agent_registered

🔍 [data_collector_1] Collecting data for: Dataset A

🔍 [data_collector_2] Collecting data for: Dataset B
💾 Stored resource: data_collector_1_collected_data
📤 [data_collector_1] Published: resource_shared
📤 [data_collector_1] Published: data_collected
💾 Stored resource: data_collector_2_collected_data
📤 [data_collector_2] Published: resource_shared
📤 [data_collector_2] Published: data_collected

✅ Parallel collection complete!


### Example 3: Agent Discovery (Simulated)

This example shows agents discovering each other by filtering the `MessageBus` messages for `agent_registered` events. In a real MCP system, discovery would happen through MCP's resource listing or tool listing endpoints. Here, it's just a list filter on in-memory Python data.

In [12]:
def discover_agents():
    """Discover all registered agents and their capabilities"""
    print("\n🔍 Agent Discovery via MCP:")
    print("="*80)

    registration_messages = [m for m in message_bus.messages
                            if m["type"] == "agent_registered"]

    for msg in registration_messages:
        agent_id = msg["agent_id"]
        content = msg["content"]
        print(f"\n🤖 Agent: {agent_id}")
        print(f"   Role: {content['role']}")
        print(f"   Capabilities: {', '.join(content['capabilities'])}")

    print("\n" + "="*80)

discover_agents()


🔍 Agent Discovery via MCP:

🤖 Agent: data_collector
   Role: Data Collection
   Capabilities: web_scraping, api_calls, data_extraction

🤖 Agent: analyzer
   Role: Data Analysis
   Capabilities: statistical_analysis, pattern_recognition, insights

🤖 Agent: report_generator
   Role: Report Generation
   Capabilities: report_writing, visualization, documentation

🤖 Agent: validator
   Role: Validation
   Capabilities: quality_check, verification, compliance

🤖 Agent: data_collector
   Role: Data Collection
   Capabilities: web_scraping, api_calls, data_extraction

🤖 Agent: data_collector
   Role: Data Collection
   Capabilities: web_scraping, api_calls, data_extraction



### Example 4: Simulated Real-time Agent Communication

This demo simulates a back-and-forth communication sequence between agents using the custom `MessageBus`. The `asyncio.sleep` calls create the illusion of real-time processing, but no MCP transport or protocol is involved — it's all in-process Python method calls.

In [13]:
async def realtime_communication_demo():
    """Demonstrate real-time inter-agent communication"""

    print("\n🔴 LIVE: Real-time Agent Communication Demo")
    print("="*80)

    # Create a new message bus for clean demo
    demo_bus = MessageBus()

    # Create agents with the demo bus
    class DemoAgent(MCPAgent):
        def __init__(self, agent_id, role):
            self.agent_id = agent_id
            self.role = role
            self.capabilities = []
            self.message_bus = demo_bus

    agent_a = DemoAgent("AgentA", "Requester")
    agent_b = DemoAgent("AgentB", "Processor")
    agent_c = DemoAgent("AgentC", "Validator")

    # Simulate conversation
    print("\n📤 AgentA: Requesting data processing")
    agent_a.publish_result("request", {"task": "Process customer data"})
    await asyncio.sleep(0.5)

    print("📥 AgentB: Received request, processing...")
    messages = agent_b.get_messages("request")
    if messages:
        agent_b.publish_result("processing", {"status": "in_progress"})
    await asyncio.sleep(0.5)

    print("📤 AgentB: Sharing processed results")
    agent_b.share_resource("processed_data", {"result": "Customer insights"})
    await asyncio.sleep(0.5)

    print("📥 AgentC: Validating results")
    resource_msgs = agent_c.get_messages("resource_shared")
    if resource_msgs:
        resource_id = resource_msgs[-1]["content"]["resource_id"]
        data = agent_c.access_resource(resource_id)
        agent_c.publish_result("validated", {"status": "approved"})

    print("\n✅ Communication sequence complete!")
    print("="*80)

    return demo_bus.messages

demo_messages = await realtime_communication_demo()


🔴 LIVE: Real-time Agent Communication Demo

📤 AgentA: Requesting data processing
📤 [AgentA] Published: request
📥 AgentB: Received request, processing...
📤 [AgentB] Published: processing
📤 AgentB: Sharing processed results
💾 Stored resource: AgentB_processed_data
📤 [AgentB] Published: resource_shared
📥 AgentC: Validating results
📤 [AgentC] Published: validated

✅ Communication sequence complete!


## Part 6: Why MCP Would Be Beneficial (Conceptual Motivation)

The patterns demonstrated in this simulation (message passing, resource sharing, tool discovery) are exactly the kinds of problems that the real MCP protocol solves. If this system were built on actual MCP, it would gain these **key advantages**:

1. **Standardized Communication**: All agents would use the same JSON-RPC-based protocol with well-defined message types
2. **Resource Sharing**: MCP's resource protocol would replace our ad-hoc `MessageBus.resources` dict with a typed, URI-based system
3. **Loose Coupling**: Agents would interact only through MCP's defined interfaces — no shared Python objects in memory
4. **Discoverability**: MCP's `tools/list` and `resources/list` endpoints would replace our manual discovery code
5. **Scalability**: Agents could run as separate processes or services connected via stdio/SSE, not just in-process
6. **Interoperability**: Any MCP-compatible client (Claude Desktop, IDEs, etc.) could interact with the agents

## Part 7: Simulated Tool Registry (Advanced Pattern)

The `MCPToolRegistry` below demonstrates the *concept* of agents exposing and discovering tools. In this simulation, tools are stored as plain Python callables in a dictionary. In a real MCP implementation, tools would be exposed via `mcp.server.Server` with JSON schemas and invoked through the MCP client protocol.

In [ ]:
class MCPToolRegistry:
    """Simulated tool registry using a plain Python dict of callables.
    In a real MCP system, tools would be exposed via mcp.server.Server with JSON
    schemas and invoked through the MCP client protocol (JSON-RPC over stdio/SSE)."""

    def __init__(self):
        self.tools: Dict[str, Dict] = {}

    def register_tool(self, agent_id: str, tool_name: str, tool_func, description: str):
        """Register a tool exposed by an agent"""
        tool_id = f"{agent_id}.{tool_name}"
        self.tools[tool_id] = {
            "agent_id": agent_id,
            "name": tool_name,
            "function": tool_func,
            "description": description
        }
        print(f"🔧 Registered tool: {tool_id}")

    def discover_tools(self, capability: Optional[str] = None) -> List[Dict]:
        """Discover available tools"""
        if capability:
            return [t for t in self.tools.values()
                   if capability.lower() in t["description"].lower()]
        return list(self.tools.values())

    def call_tool(self, tool_id: str, *args, **kwargs):
        """Call a registered tool"""
        if tool_id in self.tools:
            return self.tools[tool_id]["function"](*args, **kwargs)
        raise ValueError(f"Tool {tool_id} not found")

# Create global tool registry
tool_registry = MCPToolRegistry()

# Example: Register some tools
def calculate_mean(data: List[float]) -> float:
    return sum(data) / len(data)

def format_report(title: str, data: Dict) -> str:
    return f"# {title}\n\n{json.dumps(data, indent=2)}"

tool_registry.register_tool("analyzer", "calculate_mean", calculate_mean, "Calculate mean of data")
tool_registry.register_tool("report_generator", "format_report", format_report, "Format data as report")

print("\n🔍 Available MCP Tools:")
for tool in tool_registry.discover_tools():
    print(f"  - {tool['agent_id']}.{tool['name']}: {tool['description']}")

🔧 Registered tool: analyzer.calculate_mean
🔧 Registered tool: report_generator.format_report

🔍 Available MCP Tools:
  - analyzer.calculate_mean: Calculate mean of data
  - report_generator.format_report: Format data as report


## Summary

> ⚠️ **Reminder:** This notebook is a **conceptual simulation**. No actual MCP protocol communication occurs — all agent interaction is through plain Python in-memory data structures.

### What this notebook demonstrated (as a simulation):

1. **Coordinator + Specialized Workers Pattern**: A coordinator orchestrating specialized agents through a shared message bus
2. **Simulated Message Bus**: A central in-memory pub/sub system standing in for MCP-style communication
3. **Simulated Resource Sharing**: Agents sharing data via a Python dictionary (conceptually similar to MCP resources)
4. **Simulated Tool Discovery**: A registry of Python callables (conceptually similar to MCP tool listing)
5. **Workflow Orchestration**: Sequential and parallel agent execution using `asyncio`
6. **Inter-agent Messaging**: Publish/subscribe pattern for agent-to-agent communication

### What was NOT demonstrated:

- **Actual MCP transport** (stdio, SSE, or HTTP)
- **MCP JSON-RPC messaging** (the real protocol format)
- **MCP Server instances** (`mcp.server.Server` was imported but never used)
- **MCP Client connections** (no `mcp.client` usage)
- **Cross-process agent communication** (everything runs in a single Python process)

### Real-world Applications (if built on real MCP):

- **Data Processing Pipelines**: Specialized agents as MCP servers for ETL workflows
- **Autonomous Systems**: Agents coordinating via standardized MCP interfaces
- **AI Assistants**: Multiple specialized AI agents collaborating through MCP
- **Microservices**: Agent-based architectures with MCP as the communication layer
- **Distributed Computing**: Agents running as separate processes/services connected via MCP transport

### Next Steps to Use Real MCP:

1. Replace `MessageBus` with actual `mcp.server.Server` instances exposing tools and resources
2. Use `mcp.client` to connect agents via stdio or SSE transport
3. Define proper JSON schemas for tool inputs/outputs
4. Run agents as separate processes for true distributed operation
5. Add authentication using MCP's built-in security features
6. Implement persistent state with MCP resource URIs